# 03 — Tries and Constrained Decoding
## Turning "only these outputs are allowed" into a fast lookup

**Audience:** generalist engineering students with varied backgrounds
**Format:** context → mental model → worked examples → checks → open project
**Rule:** every instructional example is complete and executable. Only the final project is intentionally unfinished.

### How to use this notebook

Run the cells from top to bottom. Every code cell is finished and runnable. Only the last cell (the project) is left for you.

You will see:

- **Predict** — before you run a cell, write down what you expect it to print, and why.
- **What you just saw** — a short note after a cell.
- **`assert` lines** — the specification. If one fails after you edit a cell, your change broke a rule.

The project at the end is open.

In [1]:
from __future__ import annotations

import random
from dataclasses import dataclass, field
from typing import Sequence

SEED = 7
random.seed(SEED)
print(f"Ready. Seed = {SEED}")

Ready. Seed = 7


## Before we start: the problem

A language model writes one token at a time. Normally, at each step, *any* token in the vocabulary is a candidate.

Sometimes only a fixed set of outputs is legal — tool names, product codes, the keywords of a grammar. **Constrained decoding** means: at each step, only let the model pick a token that can still lead to a legal output.

A **trie** (say "try") is the data structure for that. It is a tree organised by shared beginnings. The words `run`, `read`, `reset` all start with `r`; `read` and `reset` also share `re`. Store them once, following the shared path.

We use integer token IDs, not letters, because a real model works in tokenizer IDs. The trie answers two **different** questions:

- `allowed_next(prefix)` — given what has been produced so far, which token IDs may come next?
- `contains(sequence)` — is this exact sequence a complete, legal output?

These are different because one legal command can be the start of another. Then the mask step turns the trie's answer into a decision: give every illegal token a score of `-inf` so it cannot be chosen.

## 1. Why shared beginnings matter

If the legal outputs share beginnings, a trie answers "what can come next?" in time proportional to the length of the prefix — it does **not** scan the whole list of legal outputs.

**Predict.** We insert three sequences: `[101, 12, 45]`, `[101, 12, 88]`, `[101, 90]`.
- What does `allowed_next([101])` return?
- What does `allowed_next([101, 12])` return?

![Trie structure](assets/trie.svg)

In [2]:
@dataclass
class TrieNode:
    children: dict[int, "TrieNode"] = field(default_factory=dict)
    terminal: bool = False          # True = a legal sequence ends here

class TokenTrie:
    def __init__(self) -> None:
        self.root = TrieNode()

    def insert(self, sequence: Sequence[int]) -> None:
        node = self.root
        for token in sequence:
            node = node.children.setdefault(token, TrieNode())
        node.terminal = True

    def _walk(self, prefix: Sequence[int]) -> "TrieNode | None":
        node = self.root
        for token in prefix:
            node = node.children.get(token)
            if node is None:
                return None          # this prefix is not in the trie
        return node

    def allowed_next(self, prefix: Sequence[int]) -> set[int]:
        node = self._walk(prefix)
        return set() if node is None else set(node.children)

    def contains(self, sequence: Sequence[int]) -> bool:
        node = self._walk(sequence)
        return bool(node and node.terminal)

In [3]:
trie = TokenTrie()
for sequence in ([101, 12, 45], [101, 12, 88], [101, 90]):
    trie.insert(sequence)

print("allowed_next([101])    :", trie.allowed_next([101]))
print("allowed_next([101, 12]):", trie.allowed_next([101, 12]))
print("contains([101, 12, 45]):", trie.contains([101, 12, 45]))
print("contains([101, 12])    :", trie.contains([101, 12]), " <- a real node, but no sequence ends there")

assert trie.allowed_next([101]) == {12, 90}
assert trie.allowed_next([101, 12]) == {45, 88}
assert trie.contains([101, 12, 45])
assert not trie.contains([101, 12])

allowed_next([101])    : {90, 12}
allowed_next([101, 12]): {88, 45}
contains([101, 12, 45]): True
contains([101, 12])    : False  <- a real node, but no sequence ends there


In [4]:
trie.insert([101, 12])   # now [101, 12] is ALSO a complete sequence

print("contains([101, 12])    :", trie.contains([101, 12]), " <- now True")
print("allowed_next([101, 12]):", trie.allowed_next([101, 12]), " <- still has children 45 and 88")
print("allowed_next([999])    :", trie.allowed_next([999]), " <- unknown prefix -> empty set")

assert trie.contains([101, 12])
assert trie.allowed_next([101, 12]) == {45, 88}
assert trie.allowed_next([999]) == set()

contains([101, 12])    : True  <- now True
allowed_next([101, 12]): {88, 45}  <- still has children 45 and 88
allowed_next([999])    : set()  <- unknown prefix -> empty set


### What you just saw

Think of the current prefix as a **state**. Each child is a legal move to the next state. `terminal=True` means "you are allowed to stop here" — it does **not** mean "there are no moves left".

So `contains([101, 12])` and `allowed_next([101, 12])` really are two questions: after we also inserted `[101, 12]`, that node is both a valid stopping point *and* has continuations `45` and `88`.

## 2. Cost

Looking up a prefix of length `L` costs `O(L)` — you walk `L` steps down the tree. It does not depend on how many sequences are stored. Memory is proportional to the number of distinct nodes.

Character tries are for teaching; a real constrained decoder must work on the model's token IDs, because that is what the model actually chooses.

## 3. From "allowed set" to a score mask

The trie gives you a set of allowed token IDs. The decoder works with **logits** (one score per vocabulary token). So convert: keep the allowed logits, set every other logit to `-inf`. After `softmax`, the `-inf` entries get probability exactly 0 and the allowed tokens keep their relative probabilities.

**Predict.** `logits = [0.2, 1.4, -0.1, 2.1, 0.7]`, allowed set `{1, 4}`. Which index is the arg-max of the masked logits?

In [5]:
import numpy as np

def mask_logits(logits: np.ndarray, allowed: set[int]) -> np.ndarray:
    logits = np.asarray(logits, dtype=float)
    if not allowed:
        raise ValueError("No legal continuation")              # a dead end, not a distribution
    if min(allowed) < 0 or max(allowed) >= logits.size:
        raise IndexError("allowed token outside vocabulary")
    masked = np.full_like(logits, -np.inf)                      # start: everything forbidden
    indices = np.fromiter(allowed, dtype=int)
    masked[indices] = logits[indices]                           # copy back the allowed ones
    return masked

logits = np.array([0.2, 1.4, -0.1, 2.1, 0.7])
masked = mask_logits(logits, {1, 4})

print("original logits:", logits)
print("masked logits  :", masked)
print("arg-max of masked:", int(np.argmax(masked)), " <- index 1 (2.1 at index 3 is forbidden)")

assert np.argmax(masked) == 1

original logits: [ 0.2  1.4 -0.1  2.1  0.7]
masked logits  : [-inf  1.4 -inf -inf  0.7]
arg-max of masked: 1  <- index 1 (2.1 at index 3 is forbidden)


In [6]:
# The forbidden positions are -inf; the allowed positions are untouched.
print("positions 0,2,3 are -inf:", np.isneginf(masked[[0, 2, 3]]).all())
print("positions 1,4 unchanged  :", np.array_equal(masked[[1, 4]], logits[[1, 4]]))

# Bad inputs raise instead of returning nonsense.
for bad in (set(), {-1}, {logits.size}):
    try:
        mask_logits(logits, bad)
    except (ValueError, IndexError) as exc:
        print(f"mask_logits(logits, {bad!s:10}) -> {type(exc).__name__}: {exc}")
    else:
        raise AssertionError(f"Expected a failure for {bad}")

positions 0,2,3 are -inf: True
positions 1,4 unchanged  : True
mask_logits(logits, set()     ) -> ValueError: No legal continuation
mask_logits(logits, {-1}      ) -> IndexError: allowed token outside vocabulary
mask_logits(logits, {5}       ) -> IndexError: allowed token outside vocabulary


### Why `-inf` and not, say, a big negative number

`softmax` turns logits into probabilities. `exp(-inf) == 0` exactly, so a `-inf` logit gets probability 0 and the allowed tokens keep their relative sizes. A merely "very negative" number would still leave a tiny probability.

An empty allowed set means the prefix is a dead end — there is no valid next token. That is a decoding failure, so `mask_logits` raises. A real decoder might instead emit an end-of-sequence token or back up a step.

### One full constrained step

Put the two halves together. Given the tokens produced so far: ask the trie for the legal next tokens, mask the model's logits to that set, take the arg-max. The model is never allowed to pick an illegal token, however much it "wants" to.

**Predict.** The model's logits below peak hard on token 88. At prefix `[101, 12]` the legal set is `{45, 88}`. Which token does `decode_step` return? What happens at the dead-end prefix `[101, 12, 45]`?

In [7]:
def decode_step(trie: TokenTrie, prefix: Sequence[int], logits: np.ndarray) -> int:
    allowed = trie.allowed_next(prefix)
    masked = mask_logits(logits, allowed)      # raises if the prefix is a dead end
    return int(np.argmax(masked))

step_trie = TokenTrie()
for sequence in ([101, 12, 45], [101, 12, 88], [101, 90]):
    step_trie.insert(sequence)

rng = np.random.default_rng(0)
raw_logits = rng.normal(size=128)
raw_logits[88] = 10.0     # the model strongly wants token 88
raw_logits[45] = 5.0

first = decode_step(step_trie, [], raw_logits)
print("first token (legal set {101, 90}):", first, " <- not 88, whatever the logits say")

second = decode_step(step_trie, [101, 12], raw_logits)
print("after [101, 12] (legal set {45, 88}):", second, " <- 88 is legal here AND scores highest")
assert second == 88

try:
    decode_step(step_trie, [101, 12, 45], raw_logits)   # terminal, no children
except ValueError as exc:
    print("after [101, 12, 45]:", type(exc).__name__, "-", exc, " <- dead end, correctly refused")
else:
    raise AssertionError("a dead-end prefix must not yield a token")

first token (legal set {101, 90}): 101  <- not 88, whatever the logits say
after [101, 12] (legal set {45, 88}): 88  <- 88 is legal here AND scores highest
after [101, 12, 45]: ValueError - No legal continuation  <- dead end, correctly refused


## 4. Deleting a sequence, and pruning

To delete a sequence: clear its `terminal` flag. Then walk back up and remove nodes that are now useless — but **keep** any node that is still `terminal` or still has children, because another command may share that path.

In [8]:
def delete(trie: TokenTrie, sequence: Sequence[int]) -> bool:
    path: list[tuple[TrieNode, int]] = []
    node = trie.root
    for token in sequence:
        if token not in node.children:
            return False                    # not present: nothing to do
        path.append((node, token))
        node = node.children[token]
    if not node.terminal:
        return False                        # the path exists but is not a complete sequence
    node.terminal = False
    for parent, token in reversed(path):    # walk back toward the root
        child = parent.children[token]
        if child.terminal or child.children:
            break                           # still needed by something else -> stop pruning
        del parent.children[token]
    return True

removed = delete(trie, [101, 12, 45])
print("delete([101, 12, 45]) ->", removed)
print("contains([101, 12, 88]) still True:", trie.contains([101, 12, 88]), " <- sibling survived")
assert removed and trie.contains([101, 12, 88])

delete([101, 12, 45]) -> True
contains([101, 12, 88]) still True: True  <- sibling survived


In [9]:
deletion_trie = TokenTrie()
for sequence in ([101, 12], [101, 12, 88], [101, 90]):
    deletion_trie.insert(sequence)

print("delete a sequence that isn't there:", delete(deletion_trie, [101, 12, 45]))
print("delete([101, 12]) (it has a child 88):", delete(deletion_trie, [101, 12]))
print("  contains([101, 12])    ->", deletion_trie.contains([101, 12]), " (flag cleared)")
print("  contains([101, 12, 88])->", deletion_trie.contains([101, 12, 88]), " (child kept: node not pruned)")

assert not delete(deletion_trie, [101, 12, 45])
assert not deletion_trie.contains([101, 12])
assert deletion_trie.contains([101, 12, 88])

delete a sequence that isn't there: False
delete([101, 12]) (it has a child 88): True
  contains([101, 12])    -> False  (flag cleared)
  contains([101, 12, 88])-> True  (child kept: node not pruned)


### What you just saw

Deletion has two parts: clear the `terminal` flag for the chosen sequence, then delete nodes that no longer serve **any** sequence. `[101, 12]` kept its node because `[101, 12, 88]` still needs it.

Deleting a sequence that is not there returns `False` and changes nothing — safe to call twice.

## Project — Constrained command decoder

Design a tiny tokenizer and constrain decoding to a registry of command strings. Support insertion, deletion, EOS, masked greedy decoding and useful dead-end diagnostics.

**Suggested test matrix:**

- Commands with shared prefixes remain independently discoverable.
- A command may be both complete and a prefix of another command.
- Inserting the same command twice does not create duplicate behavior.
- An invalid prefix is rejected or produces a clear empty continuation set.
- Illegal logits become `-inf`; legal logits keep their original values.
- An empty allowed-token set produces a useful dead-end error or an explicit EOS fallback.
- Deleting a command preserves its siblings and shared prefixes; deleting it again is harmless.
- Tokenizer boundaries are used consistently: decoding must operate on token IDs, not raw characters.
- Greedy decoding never selects a token outside the trie constraint.

**Acceptance criteria:** shared prefixes work; a command may be the prefix of another; invalid prefixes fail clearly; tests cover deletion and EOS; complexity is explained.

You may `from course_utils import TokenTrie, mask_logits, decode_step` instead of copying the cells above; the module ships the same implementations with docstrings.

**Checks to run yourself**

- Decode a full command greedily under the constraint and assert every emitted token was in `allowed_next` at that step.
- Give the decoder logits that peak on an illegal token and confirm it still emits a legal one.
- Reach a state whose only legal continuation is EOS and confirm your explicit EOS policy fires (not a crash).
- Delete a command mid-session and re-run a decode that shared its prefix; the sibling command must still decode.
- Feed a prefix the tokenizer never produces and check the diagnostic names the failing prefix.

In [ ]:
# PROJECT WORKSPACE — intentionally incomplete
#
# Reuse the trie + masking code from this notebook, or import it:
#     from course_utils import TokenTrie, mask_logits, decode_step
#
# 1. Tokenizer: map each command string to a list of integer token IDs and back.
#    Decoding MUST work on IDs, not characters.
# 2. Build a TokenTrie over the tokenised commands (+ an EOS id).
# 3. greedy_decode(logits_fn): step with decode_step; stop on EOS; never exceed a max length.
# 4. Dead-end policy: raise a diagnostic that names the failing prefix, OR fall back to EOS
#    when EOS is legal. Document which.
# 5. Tests: the suggested test matrix above.

class ConstrainedCommandDecoder:
    ...


raise NotImplementedError("Implement the constrained command decoder")
